In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

In [ ]:
df = pd.read_csv('../source/csv/FReDA4.csv')
df = df[df["Role"] == "Anchor"].dropna()

In [ ]:
print('5th',df['Relationship Length (years)'].quantile(0.05))
print('25th',df['Relationship Length (years)'].quantile(0.25))
print('50th',df['Relationship Length (years)'].quantile(0.5))
print('75th',df['Relationship Length (years)'].quantile(0.75))
print('95th',df['Relationship Length (years)'].quantile(0.95))

In [ ]:
df['Age'].max()

In [ ]:
def len_to_bins(val):
    if val <= 2:
        return "Short (up to 2 years)"
    elif val <= 5:
        return "Medium (2 to 5 years)"
    elif val <= 10:
        return "Mediun (5 to 10 years)"
    elif val <= 17:
        return "Long (10 to 17 years)"
    else:
        return "Very Long (more than 17 years)"

df['Relationship Length (bins)'] = df['Relationship Length (years)'].apply(len_to_bins)

In [ ]:
df.groupby(by=["Relationship Length (bins)"]).count()/ len(df) * 100

In [ ]:
def age_to_bins(val):
    if val <= 21:
        # return "<21 years old"
        return 1
    elif val <= 25:
        # return "21 to 25 years old"
        return 2
    elif val <= 30:
        # return "26 to 30 years old"
        return 3
    elif val <= 35:
        # return "31 to 35 years old"
        return 4
    elif val <= 40:
        # return "36 to 40 years old"
        return 5
    elif val <= 45:
        # return "41 to 45 years old"
        return 6
    elif val <= 50:
        # return "46 to 50 years old"
        return 7
    else:
        # return ">50 years old"
        return 8

df['Age (bins)'] = df['Age'].apply(age_to_bins)

In [ ]:
df.groupby(by=["Age (bins)"]).count()/ len(df) * 100

In [ ]:
def count_to_size(count):
    if count <= 50:
        return 250
    elif count <= 100:
        return 500
    elif count <= 200:
        return 750
    elif count <= 300:
        return 1000
    elif count <= 400:
        return 1250
    else:
        return 1500

In [ ]:
palette = [
    # '#edf8b1',
    '#c7e9b4',
    '#7fcdbb',
    '#41b6c4',
    '#1d91c0',
    '#225ea8'
]

avoid_palette = [
    "#C4A000",
    "#355C9C",
    "#B76A8D",
    "#009688",
]

In [ ]:
target_x = 'Age (bins)'

options = [
    "Short (up to 2 years)",
    "Medium (2 to 5 years)",
    "Mediun (5 to 10 years)",
    "Long (10 to 17 years)",
    "Very Long (more than 17 years)"
]

fig, ax = plt.subplots(figsize=(12, 8))

for i, option in enumerate(options):
    color = palette[i]
    data = df[df['Relationship Length (bins)'] == option]

    sns.regplot(
        data=data,
        x=target_x,
        y="Frequency",
        scatter=False,
        ax=ax,
        color=color
    )

    binned_data = data.groupby(target_x, observed=False).agg(
        x_mean=(target_x, 'mean'),
        y_mean=('Frequency', 'mean'),
        point_count=('Frequency', 'count')
    ).dropna()

    binned_data['marker_size'] = binned_data['point_count'].apply(count_to_size)

    ax.scatter(
        x=binned_data['x_mean'],
        y=binned_data['y_mean'],
        s=binned_data['marker_size'],
        color=color,
        alpha=0.65,
    )

font_axis = 32
font_ticks = 24
font_legend = 16

ax.legend(title='Relationship Length (bins)',

          loc="center",
          bbox_to_anchor=(.91, 0.5)
          )

plt.xlabel("Age", fontsize=font_axis - 4)
plt.ylabel("Frequency of Touch", fontsize=font_axis, labelpad=5)
plt.xticks([1, 2, 3, 4, 5, 6, 7, 8],
           labels=['>21', '21-25', '25-30', '30-35', '35-40',
                   '40-45', '45-50', '>50'],
           fontsize=font_ticks)

plt.yticks([0, 1, 2, 3, 4, 5],
           labels=['Not at all', '1-5\ntimes', '6-10\ntimes', '11-20\ntimes', '21-50\ntimes',
                   'More than\n 50 times'],
           rotation=0,
           fontsize=font_ticks
           );

plt.tight_layout()
plt.grid(False)
plt.show()